In [1]:
# Full, end-to-end hybrid recommendation pipeline (SVD++ + TF-IDF content)
# Includes: hyperparam tuning, alpha sweep (0.00..0.95 step 0.05), final retrain,
# evaluation of Hybrid (best alpha), Content-only, SVD++, ItemCF, Popularity,
# saves metrics CSVs and top-20 predictions per model, and creates research-oriented plots.
#
# NOTE: This file is intentionally very explicit and has many "I am doing ..." comments
# to make each step clear and intuitive for research replication.

import os
import re
import math
import random
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf

# Try to make TF behave as needed for libreco (if using TF2)
try:
    tf.compat.v1.disable_eager_execution()
except Exception:
    pass

# Libreco imports (assumes libreco is installed)
from libreco.data import random_split, DatasetPure
from libreco.algorithms import SVDpp, ItemCF

# -------------------------- Configuration -----------------------------------
# I am setting the base directories and file locations here; change BASE_DIR if needed.
BASE_DIR = r"C:\Users\GSI\Pictures\DATASETS"   # <-- I am storing dataset/result files here
DATA_DIR = os.path.join(BASE_DIR, "data")      # I am expecting CSVs in this folder
RESULTS_DIR = os.path.join(BASE_DIR, "results")# I am storing outputs here
PLOT_DIR = os.path.join(RESULTS_DIR, "plots")  # I am storing plots here

# I am pointing to the specific CSVs (adjust names/paths if different)
TRAIN_INTERACTIONS = os.path.join(DATA_DIR, "train_interactions.csv")
TEST_IN_FILE = os.path.join(DATA_DIR, "test_interactions_in.csv")
GAMES_FILE = os.path.join(DATA_DIR, "games.csv")

# I am preparing output filenames where final results will be saved
OUTPUT_TOP20_DIR = os.path.join(RESULTS_DIR, "top20_per_model")  # -> per-model top20 CSVs
EVAL_SUMMARY_OUT = os.path.join(RESULTS_DIR, "evaluation_summary_final.csv")
CSV_METRICS_OUT = os.path.join(RESULTS_DIR, "hybrid_alpha_metrics.csv")
BEST_HYBRID_TOP20 = os.path.join(RESULTS_DIR, "top20_hybrid_best_alpha.csv")
# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(OUTPUT_TOP20_DIR, exist_ok=True)

# I am setting seed and core params
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

K = 20                                        # I am using Recall@20 / NDCG@20 as main metrics
ALPHAS_GRID = np.arange(0.00, 0.96, 0.05)     # I am sweeping alpha from 0.00 to 0.95 step 0.05
CAND_CF = 500
CAND_CB = 400
RRF_K = 60
MIN_TAG_ITEMS = 3
MIN_PROFILE_NORM = 0.10

CPU_CORES = max(1, os.cpu_count() or 1)
TF_SESS_CONFIG = {"intra_op_parallelism_threads": CPU_CORES, "inter_op_parallelism_threads": CPU_CORES}

# -------------------------- Utilities --------------------------------------
# I am defining helper functions used throughout the pipeline

DELIMS_RE = re.compile(r"[\,\|/;]+")
PHRASE_SYNONYMS = {
    "fps": "first_person_shooter", "tps": "third_person_shooter",
    "rpg": "role_playing", "arpg": "action_rpg",
    "roguelike": "rogue_like", "rogue_like": "rogue_like", "rogue_lite": "rogue_lite",
}

def load_csv(path):
    # I am loading a CSV and printing a short summary line
    print(f"Loading {path}...")
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {len(df):,} rows, columns: {list(df.columns)[:8]}{'...' if len(df.columns) > 8 else ''}")
    return df

def detect_columns(df):
    # I am trying to find columns for user, item and optionally a playtime/label
    cols = [c.lower() for c in df.columns]
    def pick(cands):
        for cand in cands:
            if cand in cols:
                return df.columns[cols.index(cand)]
        return None
    return (
        pick(("user_id", "user", "userid", "user id", "steamid")),
        pick(("item_id", "itemid", "item", "game_id", "appid", "app_id", "id")),
        pick(("playtime", "play_time", "play_minutes", "playtime_forever", "rating", "label"))
    )

def normalize_phrase(s: str) -> str:
    # I am normalizing single tag/genre tokens for TF-IDF
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", "", s).replace("-", "_").replace(" ", "")
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return PHRASE_SYNONYMS.get(s, s) if s else ""

def parse_tags_or_genres(raw: str) -> list:
    # I am splitting a raw tag/genre string into normalized tokens
    if pd.isna(raw) or raw is None:
        return []
    parts = [p for p in DELIMS_RE.split(str(raw)) if p and p.strip()]
    toks, seen = [], set()
    for p in parts:
        tok = normalize_phrase(p)
        if tok and tok not in seen:
            seen.add(tok)
            toks.append(tok)
    return toks

def recall_ndcg_hr_at_k(topk_items, gt_set, k=K):
    # I am computing recall@k, ndcg@k and hit-rate@k for a single user
    if not gt_set:
        return 0.0, 0.0, 0.0
    hits = [1 if int(it) in gt_set else 0 for it in topk_items]
    recall = sum(hits) / float(len(gt_set))
    hr = 1.0 if sum(hits) > 0 else 0.0
    dcg = sum(rel / math.log2(r + 2) for r, rel in enumerate(hits))
    ideal_gains = [1] * min(len(gt_set), k)
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal_gains))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return recall, ndcg, hr

def pairwise_cosine_from_tfidf(rows):
    # I am computing one-minus-mean-pairwise-cosine as a simple intra-list diversity proxy
    if rows.shape[0] < 2:
        return 0.0
    sim = rows @ rows.T
    sim = sim.toarray()
    iu = np.triu_indices(sim.shape[0], k=1)
    return float(1.0 - sim[iu].mean())

def sigmoid(x):
    # I am using sigmoid for alpha adaptation
    return 1.0 / (1.0 + np.exp(-x))

def score_to_rrf(scores, k_rrf=RRF_K):
    # I am converting raw scores into RRF-style rank reciprocal scores
    s = np.asarray(scores).astype(float)
    if s.size == 0:
        return np.array([])
    if np.any(np.isinf(s)) or np.all(np.isnan(s)):
        # I am sanitizing invalid score arrays
        s = np.nan_to_num(s, nan=-np.inf, posinf=-np.inf, neginf=-np.inf)
    if s.size == 1:
        return np.array([1.0])
    if np.all(np.isfinite(s)) and (s.max() == s.min()):
        return np.ones_like(s) / float(k_rrf + 1.0)
    order = np.argsort(-s, kind="mergesort")
    rrf = np.zeros_like(s, dtype=float)
    for r, pos in enumerate(order):
        rrf[pos] = 1.0 / float(k_rrf + r + 1)
    if rrf.max() > 0:
        rrf = rrf / rrf.max()
    return rrf

def rrf_blend_from_scores(cf_scores, cb_scores, alpha):
    # I am blending CF and CB via normalized RRF ranks and alpha weight
    cf_rrf = score_to_rrf(cf_scores)
    cb_rrf = score_to_rrf(cb_scores)
    if cf_rrf.size == 0 and cb_rrf.size == 0:
        return np.array([])
    if cf_rrf.size == 0:
        return cb_rrf
    if cb_rrf.size == 0:
        return cf_rrf
    return alpha * cf_rrf + (1.0 - alpha) * cb_rrf

def conservative_alpha(u, base_alpha, user_prof_norm_map, user_tagged_counts):
    # I am adjusting alpha toward CF when the user's content profile is weak
    prof_norm = float(user_prof_norm_map.get(int(u), 0.0))
    tagged_cnt = int(user_tagged_counts.get(int(u), 0))
    if prof_norm <= 0 or tagged_cnt < MIN_TAG_ITEMS:
        deficit = (MIN_TAG_ITEMS - min(tagged_cnt, MIN_TAG_ITEMS)) / float(max(1, MIN_TAG_ITEMS))
        bump = 0.5 + 0.5 * deficit
        return float(min(1.0, base_alpha + (1.0 - base_alpha) * bump))
    factor = sigmoid((prof_norm - MIN_PROFILE_NORM) * 2.0)
    return float(base_alpha * (0.5 + 0.5 * factor))

# ------------------------- Data preparation ---------------------------------

def prepare_data(train_interactions_path=TRAIN_INTERACTIONS, games_path=GAMES_FILE, test_in_path=TEST_IN_FILE):
    # I am loading interaction and games data, then splitting into train/eval/test (0.8/0.1/0.1)
    print("\n--- Step 1: Load and clean datasets ---")
    inter_df = load_csv(train_interactions_path)
    u_col, i_col, p_col = detect_columns(inter_df)
    print(f"Detected -> user: {u_col}, item: {i_col}, playtime/label: {p_col}")
    inter_df = inter_df.rename(columns={u_col: "user", i_col: "item"})
    inter_df = inter_df.dropna(subset=["user", "item"])
    inter_df["user"] = inter_df["user"].astype(int)
    inter_df["item"] = inter_df["item"].astype(int)
    if p_col:
        inter_df["playtime"] = inter_df[p_col].astype(float)
    else:
        inter_df["playtime"] = 1.0
    inter_df["label"] = (inter_df["playtime"] > 0).astype(int)
    inter_df = inter_df.drop_duplicates(["user", "item"], keep="last")
    print(f"After cleaning: {len(inter_df):,} rows, users={inter_df['user'].nunique():,}, items={inter_df['item'].nunique():,}")

    games_df = load_csv(games_path)
    if os.path.exists(test_in_path):
        test_in_df = load_csv(test_in_path)
        tu, ti, _ = detect_columns(test_in_df)
        test_in_df = test_in_df.rename(columns={tu: 'user', ti: 'item'})
        test_in_df['user'] = test_in_df['user'].astype(int)
        test_in_df['item'] = test_in_df['item'].astype(int)
    else:
        test_in_df = None

    print("\n--- Step 2: Randomly splitting data (0.8/0.1/0.1) ---")
    train_df, eval_df, test_df = random_split(inter_df[['user', 'item', 'label']], multi_ratios=[0.8, 0.1, 0.1], seed=SEED)
    print(f"Split -> train={len(train_df):,}, eval={len(eval_df):,}, test={len(test_df):,}")

    print("\n--- Step 3: Build Libreco Dataset objects (train only) ---")
    train_data, data_info = DatasetPure.build_trainset(train_df)
    eval_data = DatasetPure.build_evalset(eval_df)
    test_data = DatasetPure.build_testset(test_df)

    all_items = np.asarray(data_info.item_unique_vals, dtype=int)
    all_users = np.asarray(data_info.user_unique_vals, dtype=int)
    item2pos = {int(it): i for i, it in enumerate(all_items)}

    # I am building a mapping original_user -> set(items) consumed in TRAIN only for exclusion logic
    id2user = data_info.id2user
    id2item = data_info.id2item
    user2hist_train = {}
    for inner_u, inner_item_list in data_info.user_consumed.items():
        orig_u = id2user[inner_u]
        user2hist_train[orig_u] = set(id2item[i] for i in inner_item_list)

    return dict(
        inter_df=inter_df,
        train_df=train_df,
        eval_df=eval_df,
        test_df=test_df,
        test_in_df=test_in_df,
        train_data=train_data,
        eval_data=eval_data,
        test_data=test_data,
        data_info=data_info,
        all_items=all_items,
        all_users=all_users,
        item2pos=item2pos,
        user2hist_train=user2hist_train,
        games_df=games_df
    )

# ----------------------- Content modeling ----------------------------------

def create_tfidf_matrix(games_df, all_items, plot_dir=PLOT_DIR, min_df=1):
    # I am building TF-IDF over tags+genres aligned to `all_items` (items from dataset)
    print("\n--- Step 4: Content TF-IDF (tags+genres) ---")
    cg = {c.lower(): c for c in games_df.columns}
    g_item = cg.get('item_id') or cg.get('appid') or cg.get('id') or cg.get('app_id')
    g_tags = cg.get('tags') or cg.get('tag')
    g_genre = cg.get('genres') or cg.get('genre')
    cols = [g_item] + ([g_tags] if g_tags else []) + ([g_genre] if g_genre else [])
    meta = games_df[cols].copy()
    meta = meta.rename(columns={g_item: 'item', g_tags: 'tags', g_genre: 'genres'})
    meta['item'] = meta['item'].astype(int)
    meta = meta[meta['item'].isin(all_items)].copy()
    if 'tags' in meta.columns:
        meta['toks_tags'] = meta['tags'].apply(parse_tags_or_genres)
    else:
        meta['toks_tags'] = [[]] * len(meta)
    if 'genres' in meta.columns:
        meta['toks_genres'] = meta['genres'].apply(parse_tags_or_genres)
    else:
        meta['toks_genres'] = [[]] * len(meta)

    meta_map = meta.set_index('item')
    content_docs = []
    miss_tags = miss_gen = 0
    for it in tqdm(all_items, desc="Building content docs"):
        if it in meta_map.index:
            r = meta_map.loc[it]
            tt = r['toks_tags'] if 'toks_tags' in meta_map.columns else []
            gg = r['toks_genres'] if 'toks_genres' in meta_map.columns else []
        else:
            tt, gg = [], []
        if not tt:
            miss_tags += 1
        if not gg:
            miss_gen += 1
        prefixed = [f"t:{t}" for t in tt] + [f"g:{g}" for g in gg]
        content_docs.append(" ".join(prefixed))
    print(f"Items missing tags: {miss_tags:,} / {len(all_items):,}; missing genres: {miss_gen:,} / {len(all_items):,}")

    vec = TfidfVectorizer(token_pattern=r"(?u)\b[\w:]+\b", lowercase=False, sublinear_tf=True, use_idf=True, norm='l2', min_df=min_df)
    item_content_tfidf = vec.fit_transform(content_docs).tocsr()
    print(f"TF-IDF shape: {item_content_tfidf.shape}")

    # I am plotting content coverage histogram (tokens per item)
    nz = item_content_tfidf.getnnz(axis=1)
    plt.figure(figsize=(7, 4))
    plt.hist(nz, bins=50)
    plt.xlabel("#content tokens per item")
    plt.ylabel("Count")
    plt.title("Content Coverage per Item")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "content_coverage_hist.png"))
    plt.close()

    return item_content_tfidf, vec

def create_user_content_profiles(interactions_df, all_users, item2pos, item_content_tfidf):
    # I am building user profiles from a provided interactions dataframe (train or train+eval)
    print("\n--- Step 6: Building user content profiles (from provided interactions) ---")
    wdf = interactions_df.copy()
    if 'playtime' in wdf.columns:
        wdf['w'] = np.clip(np.log1p(wdf['playtime'].astype(float)), 0.0, 10.0)
    else:
        wdf['w'] = 1.0

    user_item_w = defaultdict(list)
    for r in tqdm(wdf.itertuples(), total=len(wdf), desc="Processing interactions"):
        u, it, w = int(r.user), int(r.item), float(r.w)
        pos = item2pos.get(it, None)
        if pos is not None:
            user_item_w[u].append((pos, w))

    user_prof, user_prof_norm, user_tagged_counts = {}, {}, {}
    for u in tqdm(all_users, desc="Building user profiles"):
        pairs = user_item_w.get(int(u), [])
        if not pairs:
            user_prof[int(u)] = None
            user_prof_norm[int(u)] = 0.0
            user_tagged_counts[int(u)] = 0
            continue
        idxs = np.array([p[0] for p in pairs], dtype=int)
        ws = np.array([p[1] for p in pairs], dtype=float)
        sw = ws.sum()
        user_tagged_counts[int(u)] = int((item_content_tfidf[idxs].getnnz(axis=1) > 0).sum())
        if sw <= 0:
            user_prof[int(u)] = None
            user_prof_norm[int(u)] = 0.0
            continue
        v = (item_content_tfidf[idxs].T.dot(ws) / (sw + 1e-12)).ravel()
        n = np.linalg.norm(v)
        if n > 0:
            v = v / n
        user_prof[int(u)] = v
        user_prof_norm[int(u)] = float(n)
    return user_prof, user_prof_norm, user_tagged_counts

# ----------------------- Models & evaluation --------------------------------

def eval_model_scores(get_scores_fn, label, test_truth, train_consumed_df, item_content_tfidf, all_items, item2pos, k=K, users_sample=None):
    # I am evaluating a score-function model (scores aligned to all_items) and returning metrics
    users = list(test_truth.keys())
    if users_sample is not None and len(users) > users_sample:
        random.seed(SEED)
        users = random.sample(users, users_sample)

    # I am precomputing consumed items per user based on the provided training-like df
    user_consumed_map = defaultdict(set)
    for r in train_consumed_df[['user', 'item']].itertuples(index=False):
        user_consumed_map[int(r.user)].add(int(r.item))

    # I am computing popularity percentiles from the provided training-like df
    pop_counts = train_consumed_df['item'].value_counts().to_dict()
    all_freqs = np.array([pop_counts.get(int(it), 0) for it in all_items], dtype=float)
    if all_freqs.max() > 0:
        pop_rank = np.argsort(np.argsort(-all_freqs))
        pop_percentile = 1.0 - (pop_rank / float(len(all_freqs) - 1 + 1e-12))
    else:
        pop_percentile = np.zeros_like(all_freqs)

    recalls, ndcgs, hrs = [], [], []
    rec_items_all = []
    diversity_list = []
    pop_percentiles = []

    def exclude_consumed_idx(u_orig):
        # I am returning indices of candidate items that were NOT consumed by u_orig in the provided train-like df
        consumed = user_consumed_map.get(int(u_orig), set())
        allow = np.array([i for i, it in enumerate(all_items) if int(it) not in consumed], dtype=int)
        return allow

    def topk_from_scores(scores, allow_idx, k: int):
        # I am selecting top-k item indices from a full-scores array but restricted to allow_idx
        scores = np.asarray(scores).reshape(-1)
        allow_idx = np.asarray(allow_idx, dtype=int)
        n_items = len(all_items)
        # Normalize or expand shapes so that selection works robustly
        if scores.size != n_items:
            if scores.size == 1:
                scores = np.full(n_items, float(scores[0]), dtype=float)
            elif scores.size == allow_idx.size and allow_idx.size > 0:
                tmp = np.full(n_items, -np.inf, dtype=float)
                tmp[allow_idx] = scores
                scores = tmp
            elif scores.ravel().size == n_items:
                scores = scores.ravel()
            else:
                scores = np.full(n_items, -np.inf, dtype=float)
        if allow_idx.size == 0:
            return np.array([], dtype=int)
        take = min(k, allow_idx.size)
        if allow_idx.size == 1:
            return allow_idx.copy()
        part = np.argpartition(-scores[allow_idx], take - 1)[:take]
        cand = allow_idx[part]
        order = cand[np.argsort(-scores[cand])]
        return order[:k]

    for u in tqdm(users, desc=f"Eval scores {label} (top-{k})"):
        gt = set(test_truth.get(u, []))
        if not gt:
            continue
        allow = exclude_consumed_idx(u)
        if allow.size == 0:
            continue
        scores = get_scores_fn(u)
        topk_idx_sorted = topk_from_scores(scores, allow, k)
        if topk_idx_sorted.size == 0:
            continue
        topk_items = all_items[topk_idx_sorted]
        r, n, hr = recall_ndcg_hr_at_k(topk_items, gt, k)
        recalls.append(r)
        ndcgs.append(n)
        hrs.append(hr)
        rec_items_all.extend([int(x) for x in topk_items.tolist()])
        try:
            rows = item_content_tfidf[topk_idx_sorted]
            diversity_list.append(pairwise_cosine_from_tfidf(rows))
        except Exception:
            pass
        pp = float(pop_percentile[topk_idx_sorted].mean()) if topk_idx_sorted.size > 0 else 0.0
        pop_percentiles.append(pp)

    return {
        "model": label,
        "recall": float(np.mean(recalls) if recalls else 0.0),
        "ndcg": float(np.mean(ndcgs) if ndcgs else 0.0),
        "hitrate": float(np.mean(hrs) if hrs else 0.0),
        "coverage": len(set(rec_items_all)) / float(len(all_items)) if rec_items_all else 0.0,
        "diversity": float(np.mean(diversity_list) if diversity_list else 0.0),
        "pop_bias": float(np.mean(pop_percentiles) if pop_percentiles else 0.0),
    }

# --------------------- CF / ICF scoring helpers -----------------------------
def cf_scores_user(u, svdpp, all_items, item2pos, top_n=CAND_CF):
    # I am converting CF (SVDpp) recommendations into a score vector aligned with all_items
    if int(u) not in svdpp.data_info.user2id:
        return np.zeros(len(all_items), dtype=float)
    try:
        recs = svdpp.recommend_user(int(u), n_rec=top_n, filter_consumed=True, inner_id=False)
    except TypeError:
        recs = svdpp.recommend_user(int(u), n_rec=top_n, inner_id=False)
    arr = recs.get(int(u), np.array([], dtype=int)) if isinstance(recs, dict) else np.asarray(recs)
    scores = np.zeros(len(all_items), dtype=float)
    for r, it in enumerate(arr[:top_n]):
        pos = item2pos.get(int(it), None)
        if pos is not None:
            scores[pos] = 1.0 / float(r + 1)
    return scores

def icf_scores_user(u, itemcf, all_items, item2pos, top_n=CAND_CF):
    # I am converting ItemCF recommendations into a score vector aligned with all_items
    if int(u) not in itemcf.data_info.user2id:
        return np.zeros(len(all_items), dtype=float)
    try:
        recs = itemcf.recommend_user(int(u), n_rec=top_n, filter_consumed=True, inner_id=False)
    except TypeError:
        recs = itemcf.recommend_user(int(u), n_rec=top_n, inner_id=False)
    arr = recs.get(int(u), np.array([], dtype=int)) if isinstance(recs, dict) else np.asarray(recs)
    scores = np.zeros(len(all_items), dtype=float)
    for r, it in enumerate(arr[:top_n]):
        pos = item2pos.get(int(it), None)
        if pos is not None:
            scores[pos] = 1.0 / float(r + 1)
    return scores

def cb_scores_user(u, user_prof_map, item_content_tfidf, all_items):
    # I am computing cosine-like score = item_content_tfidf.dot(user_profile) for content-based scores
    prof = user_prof_map.get(int(u))
    if prof is None:
        return np.zeros(len(all_items), dtype=float)
    s = item_content_tfidf.dot(prof)
    return np.asarray(s).ravel()

def pop_scores_user(pop_score_array):
    # I am returning a function that ignores user and returns the same popularity scores
    return lambda u: pop_score_array

# --------------------------- Training helpers -------------------------------
def train_svdpp_model(data_info, train_data, best_params):
    # I am retraining SVD++ with chosen hyperparameters (embed_size, lr, num_neg, n_epochs)
    print("Retraining SVD++...")
    try:
        tf.compat.v1.reset_default_graph()
    except Exception:
        pass
    embed_size, lr, num_neg, n_epochs = best_params
    svdpp = SVDpp(
        task="ranking",
        data_info=data_info,
        embed_size=embed_size,
        n_epochs=n_epochs,
        lr=lr,
        batch_size=2048,
        num_neg=num_neg,
        tf_sess_config=TF_SESS_CONFIG,
        seed=SEED,
    )
    svdpp.fit(train_data, neg_sampling=True, verbose=1, shuffle=True)
    return svdpp

def train_itemcf_model(data_info, train_data):
    # I am training an ItemCF model for baseline comparisons
    print("Training ItemCF...")
    itemcf = ItemCF(
        task="ranking",
        data_info=data_info,
        sim_type="cosine",
        k_sim=50,
        store_top_k=True,
        block_size=None,
        num_threads=CPU_CORES,
        min_common=1,
        mode="invert",
        seed=SEED,
    )
    itemcf.fit(train_data, neg_sampling=True, verbose=0)
    return itemcf

def get_best_svdpp_params(train_data, eval_truth, data_info):
    # I am running a compact hyperparameter sweep for SVD++ (simple grid) using eval set
    print("\n--- Step 5: Hyperparameter Tuning for SVD++ ---")
    param_grid = {
        "embed_size": [32],   # I am keeping this compact for speed; extend if you want full search
        "lr": [0.0005],
        "num_neg": [5],
        "n_epochs": [10],
    }
    best_score = -np.inf
    best_params = None
    EVAL_SAMPLE_USERS = min(2000, len(eval_truth))

    for embed_size, lr, num_neg, n_epochs in product(param_grid["embed_size"], param_grid["lr"], param_grid["num_neg"], param_grid["n_epochs"]):
        print(f"\n>>> SVD++ trial: embed={embed_size}, lr={lr}, num_neg={num_neg}, epochs={n_epochs}")
        try:
            tf.compat.v1.reset_default_graph()
        except Exception:
            pass
        m = SVDpp(
            task="ranking",
            data_info=data_info,
            embed_size=embed_size,
            n_epochs=n_epochs,
            lr=lr,
            batch_size=2048,
            num_neg=num_neg,
            tf_sess_config=TF_SESS_CONFIG,
            seed=SEED,
        )
        m.fit(train_data, neg_sampling=True, verbose=0, shuffle=True)
        r = recommend_eval_by_model_recommend_user(m, eval_truth, users_sample=EVAL_SAMPLE_USERS)
        print(f"Trial -> recall@{K}: {r['recall']:.4f}, ndcg@{K}: {r['ndcg']:.4f}")
        if r["ndcg"] > best_score:
            best_score = r["ndcg"]
            best_params = (embed_size, lr, num_neg, n_epochs)
    print(f"\nBest SVD++ params (by ndcg@{K} on eval sample): {best_params}, ndcg={best_score:.4f}")
    return best_params

# I am keeping the original recommend_eval_by_model_recommend_user for quick baseline prints (not used for coverage/diversity)
def recommend_eval_by_model_recommend_user(model, test_truth_map, k=K, users_sample=None):
    users = [u for u in test_truth_map.keys() if int(u) in model.data_info.user2id]
    if users_sample is not None and len(users) > users_sample:
        random.seed(SEED)
        users = random.sample(users, users_sample)
    recalls, ndcgs, hrs = [], [], []
    rec_items_all = []
    for u in tqdm(users, desc=f"Eval {model.__class__.__name__} (top-{k})"):
        gt = set(test_truth_map.get(u, []))
        if not gt:
            continue
        try:
            recs = model.recommend_user(int(u), n_rec=k, filter_consumed=True, inner_id=False)
        except TypeError:
            recs = model.recommend_user(int(u), n_rec=k, inner_id=False)
        if isinstance(recs, dict):
            topk = np.asarray(recs.get(int(u), np.array([], dtype=int)))
        else:
            topk = np.asarray(recs, dtype=int)
        if topk.size == 0:
            continue
        r, n, hr = recall_ndcg_hr_at_k(topk, gt, k)
        recalls.append(r)
        ndcgs.append(n)
        hrs.append(hr)
        rec_items_all.extend(topk.tolist())
    return {
        "model": model.__class__.__name__,
        "recall": float(np.mean(recalls) if recalls else 0.0),
        "ndcg": float(np.mean(ndcgs) if ndcgs else 0.0),
        "hitrate": float(np.mean(hrs) if hrs else 0.0),
        "coverage": len(set(rec_items_all)) / float(len(model.data_info.item_unique_vals)) if rec_items_all else 0.0,
        "diversity": None,
        "pop_bias": None
    }

# ----------------------------- Main pipeline -------------------------------

if __name__ == '__main__':
    # I am preparing data (load, clean, split, create dataset objects)
    ds = prepare_data()
    inter_df = ds['inter_df']
    train_df = ds['train_df']
    eval_df = ds['eval_df']
    test_df = ds['test_df']
    test_in_df = ds['test_in_df']
    train_data = ds['train_data']
    eval_data = ds['eval_data']
    test_data = ds['test_data']
    data_info = ds['data_info']
    all_items = ds['all_items']
    all_users = ds['all_users']
    item2pos = ds['item2pos']
    user2hist_train = ds['user2hist_train']
    games_df = ds['games_df']

    # I am building TF-IDF content aligned with the initial all_items
    item_content_tfidf, tfidf_vec = create_tfidf_matrix(games_df, all_items, PLOT_DIR, min_df=1)

    # I am building user content profiles from TRAIN only (no leakage)
    user_prof_train, user_prof_norm_train, user_tagged_counts_train = create_user_content_profiles(train_df, all_users, item2pos, item_content_tfidf)

    # I am getting grouped truth maps for eval/test
    test_truth = test_df.groupby('user')['item'].apply(list).to_dict()
    eval_truth = eval_df.groupby('user')['item'].apply(list).to_dict()

    # I am tuning SVD++ hyperparams using train->eval
    best_svdpp_params = get_best_svdpp_params(train_data, eval_truth, data_info)

    # I am training SVD++ and ItemCF on TRAIN for use in alpha tuning
    svdpp_model = train_svdpp_model(data_info, train_data, best_svdpp_params)
    itemcf_model = train_itemcf_model(data_info, train_data)

    # I am computing popularity scores from TRAIN (for baseline)
    pop_counts = train_df['item'].value_counts()
    pop_scores_train = np.array([pop_counts.get(int(it), 0) for it in all_items], dtype=float)

    # -------------------- Hybrid alpha tuning (using EVAL) -------------------
    print('\n--- Step 8: Hybrid model alpha tuning (tune on EVAL) ---')
    hybrid_eval_results = []
    EVAL_SAMPLE_USERS = min(2000, len(eval_truth))
    eval_users = list(eval_truth.keys())
    random.seed(SEED)
    sample_eval_users = random.sample(eval_users, EVAL_SAMPLE_USERS) if len(eval_users) > EVAL_SAMPLE_USERS else eval_users

    # I am preparing CF/CB score functions using TRAIN (not retrained yet)
    cf_scores_fn = lambda u: cf_scores_user(u, svdpp_model, all_items, item2pos)
    cb_scores_fn = lambda u: cb_scores_user(u, user_prof_train, item_content_tfidf, all_items)

    def get_hybrid_scores_for_alpha(alpha):
        # I am returning a function that blends CF and CB scores with adaptive alpha per user
        def _get(u):
            adapted_alpha = conservative_alpha(u, alpha, user_prof_norm_train, user_tagged_counts_train)
            cf = cf_scores_fn(u)
            cb = cb_scores_fn(u)
            return rrf_blend_from_scores(cf, cb, adapted_alpha)
        return _get

    # I am sweeping alpha and evaluating on EVAL using TRAIN as the "excluded" interactions
    for alpha in tqdm(ALPHAS_GRID, desc='Tuning Hybrid Alpha'):
        h_eval_res = eval_model_scores(
            get_hybrid_scores_for_alpha(alpha),
            f'Hybrid-α={alpha:.2f}',
            eval_truth,
            train_df,  # I am excluding TRAIN items while tuning on EVAL
            item_content_tfidf,
            all_items,
            item2pos,
            k=K,
            users_sample=EVAL_SAMPLE_USERS
        )
        hybrid_eval_results.append({
            'alpha': alpha,
            'recall': h_eval_res['recall'],
            'ndcg': h_eval_res['ndcg'],
            'diversity': h_eval_res['diversity'],
            'pop_bias': h_eval_res['pop_bias'],
            'coverage': h_eval_res['coverage']
        })

    # I am saving alpha sweep results to CSV
    hybrid_df = pd.DataFrame(hybrid_eval_results)
    hybrid_df.to_csv(CSV_METRICS_OUT, index=False)
    best_alpha = float(hybrid_df.loc[hybrid_df['ndcg'].idxmax()]['alpha'])
    print(f"Optimal alpha value from tuning (on eval): {best_alpha:.2f}")

    # ---------------- Final evaluation on TEST using retrained CF on TRAIN+EVAL ---
    print('\n--- Step 9: Final evaluation of the best hybrid model on TEST ---')
    combined_train_df = pd.concat([train_df, eval_df], ignore_index=True)
    combined_train_data, combined_data_info = DatasetPure.build_trainset(combined_train_df)

    # I am retraining SVD++ on combined (train+eval)
    svdpp_final = train_svdpp_model(combined_data_info, combined_train_data, best_svdpp_params)

    # I am training ItemCF on combined (train+eval) for final evaluation
    itemcf_final = train_itemcf_model(combined_data_info, combined_train_data)

    # I am creating all_items and item2pos aligned to retrained data
    all_items_te = np.asarray(combined_data_info.item_unique_vals, dtype=int)
    item2pos_te = {int(it): i for i, it in enumerate(all_items_te)}

    # I am rebuilding content docs aligned to retrained all_items_te but using the original TF-IDF vocabulary
    content_docs_te = []
    cg = {c.lower(): c for c in games_df.columns}
    g_item = cg.get('item_id') or cg.get('appid') or cg.get('id') or cg.get('app_id')
    g_tags = cg.get('tags') or cg.get('tag')
    g_genre = cg.get('genres') or cg.get('genre')
    cols = [g_item] + ([g_tags] if g_tags else []) + ([g_genre] if g_genre else [])
    meta = games_df[cols].copy()
    meta = meta.rename(columns={g_item: 'item', g_tags: 'tags', g_genre: 'genres'})
    meta['item'] = meta['item'].astype(int)
    meta_map = meta.set_index('item')
    for it in tqdm(all_items_te, desc='Build content docs (train+eval aligned)'):
        if it in meta_map.index:
            r = meta_map.loc[it]
            tt = parse_tags_or_genres(r['tags']) if 'tags' in meta_map.columns else []
            gg = parse_tags_or_genres(r['genres']) if 'genres' in meta_map.columns else []
        else:
            tt, gg = [], []
        prefixed = [f"t:{t}" for t in tt] + [f"g:{g}" for g in gg]
        content_docs_te.append(" ".join(prefixed))
    item_content_tfidf_te = tfidf_vec.transform(content_docs_te).tocsr()  # I am reusing the same TF-IDF vocabulary

    # I am building user profiles from combined_train_df (train+eval) for CB component
    user_prof_te, user_prof_norm_te, user_tagged_counts_te = create_user_content_profiles(combined_train_df, np.asarray(combined_data_info.user_unique_vals, dtype=int), item2pos_te, item_content_tfidf_te)

    # I am preparing final score functions aligned to retrained model and retrained item set
    cf_scores_fn_te = lambda u: cf_scores_user(u, svdpp_final, all_items_te, item2pos_te)
    icf_scores_fn_te = lambda u: icf_scores_user(u, itemcf_final, all_items_te, item2pos_te)
    cb_scores_fn_te = lambda u: cb_scores_user(u, user_prof_te, item_content_tfidf_te, all_items_te)
    pop_counts_te = combined_train_df['item'].value_counts()
    pop_scores_te = np.array([pop_counts_te.get(int(it), 0) for it in all_items_te], dtype=float)
    pop_scores_fn_te = pop_scores_user(pop_scores_te)

    # I am building the final hybrid score function that uses the selected best_alpha and per-user conservative_alpha
    def get_hybrid_scores_final(u):
        adapted = conservative_alpha(u, best_alpha, user_prof_norm_te, user_tagged_counts_te)
        cf = cf_scores_fn_te(u)
        cb = cb_scores_fn_te(u)
        return rrf_blend_from_scores(cf, cb, adapted)

    # I am evaluating Hybrid(best alpha), Content-only, SVD++, ItemCF and Popularity on the TEST set
    final_results = []
    # I am evaluating hybrid (best alpha)
    final_hybrid_res = eval_model_scores(get_hybrid_scores_final, f'Hybrid-best-α={best_alpha:.2f}', test_truth, combined_train_df, item_content_tfidf_te, all_items_te, item2pos_te, k=K)
    final_results.append(final_hybrid_res)
    print("Hybrid final:", final_hybrid_res)

    # I am evaluating content-only (CB) on TEST
    final_cb_res = eval_model_scores(cb_scores_fn_te, 'Content-only', test_truth, combined_train_df, item_content_tfidf_te, all_items_te, item2pos_te, k=K)
    final_results.append(final_cb_res)
    print("Content-only final:", final_cb_res)

    # I am evaluating SVD++ (CF) on TEST using score-conversion
    final_cf_res = eval_model_scores(cf_scores_fn_te, 'SVDpp-final', test_truth, combined_train_df, item_content_tfidf_te, all_items_te, item2pos_te, k=K)
    final_results.append(final_cf_res)
    print("SVD++ final:", final_cf_res)

    # I am evaluating ItemCF (ICF) on TEST
    final_icf_res = eval_model_scores(icf_scores_fn_te, 'ItemCF-final', test_truth, combined_train_df, item_content_tfidf_te, all_items_te, item2pos_te, k=K)
    final_results.append(final_icf_res)
    print("ItemCF final:", final_icf_res)

    # I am evaluating Popularity baseline on TEST
    final_pop_res = eval_model_scores(pop_scores_fn_te, 'Popularity-final', test_truth, combined_train_df, item_content_tfidf_te, all_items_te, item2pos_te, k=K)
    final_results.append(final_pop_res)
    print("Popularity final:", final_pop_res)

    # I am saving final summary metrics to CSV
    final_df = pd.DataFrame(final_results).set_index('model')
    final_df.to_csv(EVAL_SUMMARY_OUT)
    print(f"Saved final evaluation summary to {EVAL_SUMMARY_OUT}")
    print(final_df)

    # ---------------- Save top-20 per model (for research reproducibility) ----
    # I am saving top-20 recommendations for each model for all users present in test_in_df (if available),
    # otherwise for all users in TEST set's user list.
    if test_in_df is not None:
        users_to_pred = sorted(test_in_df['user'].unique())
    else:
        users_to_pred = sorted(test_df['user'].unique())

    # helper for writing top-k for a generic score function (aligned to all_items_te)
    def save_topk_for_model(score_fn, model_name, users_list, all_items_local, combined_train_local, item2pos_local, out_path, k_top=20):
        rows = []
        # I am precomputing consumed sets per user from combined_train_local
        consumed_map = defaultdict(set)
        for r in combined_train_local[['user', 'item']].itertuples(index=False):
            consumed_map[int(r.user)].add(int(r.item))
        for u in tqdm(users_list, desc=f"Top-{k_top} predictions ({model_name})"):
            consumed = consumed_map.get(int(u), set())
            scores = score_fn(u)
            if scores.size == 0:
                # I am falling back to zeros if no scores available
                scores = np.zeros(len(all_items_local), dtype=float)
            # I am building candidate indices excluding consumed items
            item_idxs = np.array([i for i, it in enumerate(all_items_local) if int(it) not in consumed], dtype=int)
            if item_idxs.size == 0:
                top_items = []
            else:
                take = min(k_top, item_idxs.size)
                part = np.argpartition(-scores[item_idxs], take - 1)[:take]
                cand = item_idxs[part]
                order = cand[np.argsort(-scores[cand])]
                top_items = [int(all_items_local[i]) for i in order[:k_top]]
            rows.append({'user': int(u), 'top20': top_items})
        out_df = pd.DataFrame(rows)
        out_df.to_csv(out_path, index=False)
        print(f"Saved top-{k_top} for {model_name} to {out_path}")

    # I am saving top-20 for each final model
    save_topk_for_model(get_hybrid_scores_final, f'Hybrid-best-{best_alpha:.2f}', users_to_pred, all_items_te, combined_train_df, item2pos_te, os.path.join(OUTPUT_TOP20_DIR, f"top20_hybrid_alpha_{best_alpha:.2f}.csv"))
    save_topk_for_model(cb_scores_fn_te, 'Content-only', users_to_pred, all_items_te, combined_train_df, item2pos_te, os.path.join(OUTPUT_TOP20_DIR, "top20_content.csv"))
    save_topk_for_model(cf_scores_fn_te, 'SVDpp-final', users_to_pred, all_items_te, combined_train_df, item2pos_te, os.path.join(OUTPUT_TOP20_DIR, "top20_svdpp.csv"))
    save_topk_for_model(icf_scores_fn_te, 'ItemCF-final', users_to_pred, all_items_te, combined_train_df, item2pos_te, os.path.join(OUTPUT_TOP20_DIR, "top20_itemcf.csv"))
    save_topk_for_model(pop_scores_fn_te, 'Popularity-final', users_to_pred, all_items_te, combined_train_df, item2pos_te, os.path.join(OUTPUT_TOP20_DIR, "top20_popularity.csv"))

    # -------------------- Plots for research-oriented comparisons -----------
    # I am creating several plots: Recall/NDCG bar, coverage vs diversity tradeoff scatter,
    # popularity bias bar, item popularity histogram, alpha sweep plot (recall/ndcg).
    try:
        # 1) Recall and NDCG comparison bar plot
        plt.figure(figsize=(9, 5))
        models = final_df.index.tolist()
        recalls = final_df['recall'].values
        ndcgs = final_df['ndcg'].values
        x = np.arange(len(models))
        width = 0.38
        plt.bar(x - width/2, recalls, width=width, label=f"Recall@{K}")
        plt.bar(x + width/2, ndcgs, width=width, label=f"NDCG@{K}")
        plt.xticks(x, models, rotation=30, ha='right')
        plt.ylabel("Score")
        plt.title("Recall and NDCG comparison (final models)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, "recall_ndcg_comparison.png"))
        plt.close()

        # 2) Coverage vs Diversity trade-off scatter (each model)
        plt.figure(figsize=(7, 6))
        cov = final_df['coverage'].values
        div = final_df['diversity'].values
        for i, m in enumerate(models):
            plt.scatter(cov[i], div[i], s=100)
            plt.text(cov[i], div[i], m, fontsize=9, ha='right', va='bottom')
        plt.xlabel("Coverage")
        plt.ylabel("Diversity (1 - mean cosine)")
        plt.title("Coverage vs Diversity trade-off (final models)")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, "coverage_diversity_tradeoff.png"))
        plt.close()

        # 3) Popularity bias (mean popularity percentile of top-k)
        plt.figure(figsize=(9, 4))
        pop_biases = final_df['pop_bias'].values
        plt.bar(models, pop_biases)
        plt.ylabel("Mean popularity percentile (higher = more popular items recommended)")
        plt.title("Popularity bias of final models (lower is less biased to popular items)")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, "popularity_bias.png"))
        plt.close()

        # 4) Item popularity histogram (distribution of training popularity)
        plt.figure(figsize=(7, 4))
        # I am plotting popularity histogram using combined_train_df
        counts = combined_train_df['item'].value_counts().values
        plt.hist(counts, bins=100, log=True)
        plt.xlabel("Item popularity (train+eval count)")
        plt.ylabel("Log Count")
        plt.title("Item popularity distribution (train+eval)")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, "item_popularity_hist.png"))
        plt.close()

        # 5) Alpha sweep plot (Recall and NDCG vs alpha)
        plt.figure(figsize=(8, 4))
        plt.plot(hybrid_df['alpha'], hybrid_df['recall'], marker='o', label=f"Recall@{K}")
        plt.plot(hybrid_df['alpha'], hybrid_df['ndcg'], marker='s', label=f"NDCG@{K}")
        plt.xlabel('alpha (CF weight)')
        plt.title("Hybrid alpha sweep (tuning on eval)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'alpha_sweep_recall_ndcg.png'))
        plt.close()

        # 6) Coverage and Diversity across alphas (two subplots)
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(hybrid_df['alpha'], hybrid_df['coverage'], marker='o')
        plt.xlabel('alpha')
        plt.title('Hybrid coverage vs alpha')
        plt.subplot(1, 2, 2)
        plt.plot(hybrid_df['alpha'], hybrid_df['diversity'], marker='o')
        plt.xlabel('alpha')
        plt.title('Hybrid diversity vs alpha')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, 'alpha_coverage_diversity.png'))
        plt.close()

    except Exception as e:
        print("Plotting error:", e)

    # ---------------- Save extended results and finish -----------------------
    # I am saving the best alpha's row and the final metrics as separate CSV for clarity
    best_row = hybrid_df.loc[hybrid_df['alpha'] == best_alpha]
    if not best_row.empty:
        best_row.to_csv(os.path.join(RESULTS_DIR, f"hybrid_best_alpha_{best_alpha:.2f}_metrics.csv"), index=False)
        print(f"Saved best alpha metrics for alpha={best_alpha:.2f}")

    # I am also writing the final_df (metrics for final models) already saved; additionally save JSON/CSV per-model
    final_df.to_csv(EVAL_SUMMARY_OUT)
    print("All done. Results and plots are in:", RESULTS_DIR)


Instructions for updating:
non-resource variables are not supported in the long term

--- Step 1: Load and clean datasets ---
Loading C:\Users\GSI\Pictures\DATASETS\data\train_interactions.csv...
Loaded 2,293,985 rows, columns: ['user_id', 'item_id', 'item_name', 'playtime']
Detected -> user: user_id, item: item_id, playtime/label: playtime
After cleaning: 2,267,389 rows, users=54,315, items=8,368
Loading C:\Users\GSI\Pictures\DATASETS\data\games.csv...
Loaded 8,523 rows, columns: ['item_id', 'item_name', 'publisher', 'genres', 'url', 'tags', 'sentiment', 'metascore']...
Loading C:\Users\GSI\Pictures\DATASETS\data\test_interactions_in.csv...
Loaded 448,211 rows, columns: ['user_id', 'item_id', 'item_name', 'playtime']

--- Step 2: Randomly splitting data (0.8/0.1/0.1) ---
Split -> train=1,813,911, eval=226,292, test=226,243

--- Step 3: Build Libreco Dataset objects (train only) ---

--- Step 4: Content TF-IDF (tags+genres) ---


Building content docs:   0%|          | 0/8184 [00:00<?, ?it/s]

Items missing tags: 4 / 8,184; missing genres: 385 / 8,184
TF-IDF shape: (8184, 360)

--- Step 6: Building user content profiles (from provided interactions) ---


Processing interactions:   0%|          | 0/1813911 [00:00<?, ?it/s]

Building user profiles:   0%|          | 0/53694 [00:00<?, ?it/s]

...

Saved top-20 for SVDpp-final to C:\Users\GSI\Pictures\DATASETS\results\top20_per_model\top20_svdpp.csv


Top-20 predictions (ItemCF-final):   0%|          | 0/13579 [00:00<?, ?it/s]

Saved top-20 for ItemCF-final to C:\Users\GSI\Pictures\DATASETS\results\top20_per_model\top20_itemcf.csv


Top-20 predictions (Popularity-final):   0%|          | 0/13579 [00:00<?, ?it/s]

Saved top-20 for Popularity-final to C:\Users\GSI\Pictures\DATASETS\results\top20_per_model\top20_popularity.csv
Saved best alpha metrics for alpha=0.95
All done. Results and plots are in: C:\Users\GSI\Pictures\DATASETS\results
